# 04 — Event Code Derivation

Applies the event recoding rules defined in the YAML config to produce
analysis-ready event codes from the raw trigger stream.

Rules are defined in `configs/<your_experiment>.yaml` under `event_rules`.
If your raw event codes need no transformation, skip this notebook.

**Input:** `<subject>_preprocessed_events_eve.fif`  
**Output:** `<subject>_derived_events_eve.fif`, `<subject>_event_recoding_log.txt`

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, find_subjects, derive_events_subject

# ── Update this path to point to your experiment config ──
cfg = load_config('../configs/your_experiment.yaml')

subjects = find_subjects(cfg)
print(f"Subjects: {len(subjects)}")

In [ ]:
# ── Test with the first subject ──
test_subject = subjects[0]
print(f"\nDeriving events for {test_subject}\n")

success = derive_events_subject(cfg, test_subject, overwrite=True, verbose=True)
print(f"\nResult: {'OK' if success else 'skipped or failed'}")

In [ ]:
# ── Verify: compare raw vs derived event counts ──
import mne
import numpy as np
from eeg_toolkit import get_subject_path
from eeg_toolkit.event_codes import _get_derived_events_path

events_raw     = mne.read_events(get_subject_path(cfg, test_subject, 'preprocessed_events'))
events_derived = mne.read_events(_get_derived_events_path(cfg, test_subject))

print(f"Raw events:     {len(events_raw)}")
print(f"Derived events: {len(events_derived)}")
print(f"Difference:     +{len(events_derived) - len(events_raw)} new events\n")

print("=== All event codes in derived file ===")
unique, counts = np.unique(events_derived[:, 2], return_counts=True)
for c, n in zip(unique, counts):
    tag = "  <-- NEW" if c not in np.unique(events_raw[:, 2]) else ""
    print(f"  {c}: {n}{tag}")

In [ ]:
# ── Derive events for all subjects ──
from eeg_toolkit import derive_events_all

summary = derive_events_all(cfg, overwrite=False, verbose=True)